# 00 — Environment Setup
**Paper:** *Cross-Dataset Pneumonia Detection Failure as an Ontological Problem*

Run this notebook first every Colab session. It checks GPU, mounts Drive, sets Kaggle credentials, downloads datasets (once), installs dependencies, and clones your GitHub repo.

In [ ]:
# 1. GPU CHECK
import subprocess, torch
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU. Go to Runtime > Change runtime type > GPU')
print(result.stdout)
print(f'PyTorch GPU: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0)}')

In [ ]:
# 2. MOUNT GOOGLE DRIVE + SET PATHS
from google.colab import drive
import os
drive.mount('/content/drive')

# Results, models, figures -> Drive (persists across sessions)
BASE_DIR     = '/content/drive/MyDrive/pneumonia_research'
RESULTS_DIR  = f'{BASE_DIR}/results'

# Raw image data -> Colab local disk (fast, ~80GB available, redownload each session)
DATA_DIR     = '/content/data'
KAGGLE_DIR   = f'{DATA_DIR}/kaggle'
CHEXPERT_DIR = f'{DATA_DIR}/chexpert'

for d in [BASE_DIR, RESULTS_DIR, DATA_DIR, KAGGLE_DIR, CHEXPERT_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')
print(f'  Results -> {RESULTS_DIR}  (persists)')
print(f'  Images  -> {DATA_DIR}     (local, redownloads each session)')

In [ ]:
# 3. KAGGLE CREDENTIALS
# Paste your Kaggle username and API key here.
# Do NOT commit this cell to GitHub.
import os
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'  # <-- replace
os.environ['KAGGLE_KEY']      = 'YOUR_KAGGLE_API_KEY'   # <-- replace
print('Kaggle credentials set.')

In [ ]:
# 4. INSTALL DEPENDENCIES
!pip install -q timm transformers grad-cam scikit-learn pandas matplotlib seaborn tqdm Pillow numpy scipy netcal torchmetrics
print('Dependencies installed.')

In [ ]:
# 5. DOWNLOAD DATASETS TO LOCAL COLAB DISK
# Data goes to /content/data (local, fast, ~80GB available)
# Only results/models are saved to Drive
import os
from pathlib import Path

def count_images(path):
    if not os.path.exists(path):
        return 0
    return sum(1 for p in Path(path).rglob('*')
               if p.suffix.lower() in {'.jpg','.jpeg','.png'})

# Kaggle Chest X-Ray (~1.2GB, ~2 mins)
if count_images(KAGGLE_DIR) < 1000:
    print('Downloading Kaggle Chest X-Ray (~1.2GB)...')
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia \
        -p {KAGGLE_DIR} --unzip -q
    print(f'Kaggle done. {count_images(KAGGLE_DIR):,} images.')
else:
    print(f'Kaggle already on local disk: {count_images(KAGGLE_DIR):,} images.')

# CheXpert (~14GB, ~15 mins on Colab's fast connection)
# Note: re-downloads each session since it's on local disk
# Only needs to run once per session before training
if count_images(CHEXPERT_DIR) < 10000:
    print('Downloading CheXpert (~14GB, ~15 mins)...')
    print('Tip: run this and go do something else.')
    !kaggle datasets download -d ashery/chexpert \
        -p {CHEXPERT_DIR} --unzip -q
    print(f'CheXpert done. {count_images(CHEXPERT_DIR):,} images.')
else:
    print(f'CheXpert already on local disk: {count_images(CHEXPERT_DIR):,} images.')

In [ ]:
# 6. CLONE GITHUB REPO
# Replace with your actual repo URL after creating it
import os, sys
GITHUB_REPO = 'https://github.com/YOUR_USERNAME/pneumonia-ontology-paper'
REPO_DIR    = '/content/pneumonia_research'

if os.path.exists(REPO_DIR):
    print('Repo exists, pulling latest...')
    !cd {REPO_DIR} && git pull
else:
    !git clone {GITHUB_REPO} {REPO_DIR}

sys.path.insert(0, REPO_DIR)
print('Repo ready. src/ modules importable.')

In [ ]:
# 7. SAVE GLOBAL CONFIG TO DRIVE
import json, os

CONFIG = {
    'base_dir':     BASE_DIR,
    'kaggle_dir':   KAGGLE_DIR,
    'chexpert_dir': CHEXPERT_DIR,
    'results_dir':  RESULTS_DIR,
    'chexpert_target_labels': [
        'Pneumonia', 'Lung Opacity', 'Consolidation',
        'Edema', 'Pleural Effusion', 'Atelectasis'
    ],
    'image_size':   224,
    'batch_size':   32,
    'num_workers':  4,
    'epochs':       20,
    'lr':           1e-4,
    'seed':         42,
    'biomed_model': 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext',
}

with open(f'{BASE_DIR}/config.json', 'w') as f:
    json.dump(CONFIG, f, indent=2)
print('Config saved.')
print(json.dumps(CONFIG, indent=2))

In [ ]:
# 8. SANITY CHECK
from pathlib import Path

def count_images(path):
    return sum(1 for p in Path(path).rglob('*')
               if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})

kaggle_n   = count_images(KAGGLE_DIR)
chexpert_n = count_images(CHEXPERT_DIR)

print('=== Dataset Sanity Check ===')
print(f'Kaggle images:   {kaggle_n:,}   (expected ~5,863)')
print(f'CheXpert images: {chexpert_n:,}  (expected ~220,000)')

if kaggle_n < 1000:
    print('WARNING: Kaggle looks incomplete')
if chexpert_n < 10000:
    print('WARNING: CheXpert looks incomplete')

print('\nSetup complete. Proceed to 01_main_experiment.ipynb')